# Webinar — Regresión con Machine Learning
**TripleTen · Comparando modelos de regresión**

---
## Antes de arrancar

> *Una empresa inmobiliaria entrena un modelo para predecir el precio de casas. En pruebas: R² de 0.95. Lo despliegan. En producción el modelo predice $80,000 para una mansión de $500,000.*

**¿Qué crees que salió mal? Escribe en el chat.**

---

## Agenda

| # | Bloque |
|---|--------|
| 1 | Dataset y exploración |
| 2 | Métricas de regresión: RMSE, MAE, R² |
| 3 | Pipeline + Cross Validation |
| 4 | Comparar 3 modelos |
| 5 | GridSearch |
| 6 | Feature Importances |
| 7 | Modelo final |

**Al terminar vas a poder:**
- Entender cuándo usar RMSE vs MAE vs R²
- Detectar data leakage antes de entrenar
- Construir un pipeline completo de preprocesamiento + modelo
- Comparar LinearRegression, DecisionTree y RandomForest visualmente
- Identificar qué features impulsan el precio de una propiedad

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Los 3 modelos que vamos a comparar
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.datasets import fetch_california_housing

np.random.seed(42)
print('✅ Librerías cargadas')

✅ Librerías cargadas


---
# PARTE 1 — Dataset y exploración

Usamos **California Housing** — dataset clásico incluido en scikit-learn.  
El objetivo es predecir el **precio mediano de casas** (en cientos de miles de dólares) a partir de características del vecindario.

| Feature | Descripción |
|---------|-------------|
| `MedInc` | Ingreso mediano del vecindario |
| `HouseAge` | Antigüedad mediana de las casas |
| `AveRooms` | Promedio de habitaciones por hogar |
| `AveBedrms` | Promedio de dormitorios por hogar |
| `Population` | Población del bloque |
| `AveOccup` | Promedio de ocupantes por hogar |
| `Latitude` | Latitud |
| `Longitude` | Longitud |
| `MedHouseVal` | **Target**: valor mediano de la casa |


In [ ]:
# Cargar dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame

#Que debemos revisar del dataset?

df.head()

Shape: (20640, 9)
Nulos: 0
Target (MedHouseVal): min=0.15, max=5.00, media=2.07


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [2]:
# Distribución del target y correlaciones

# Correlación de features con el target

plt.tight_layout()
plt.show()

# Nota: MedInc tiene la correlación más alta (~0.69) — pista sobre feature importance

NameError: name 'plt' is not defined

---
# PARTE 2 — Data leakage en regresión

En regresión el data leakage es más silencioso que en clasificación — el modelo sigue funcionando, pero con métricas irreales.

**Ejemplo típico en precios de casas:**
- `precio_por_m2` = `precio_final / metros_cuadrados` → usa el target para calcular la feature ❌
- `valuacion_fiscal_actualizada` → se calcula después de la venta ❌
- `historial_de_ventas_reciente` → no existe al momento de predecir ❌

**Regla de oro:** antes de meter una columna al modelo, preguntá —  
*¿Esta información estaría disponible en el momento real de la predicción?*


---
# PARTE 3 — Métricas de regresión

A diferencia de clasificación (accuracy, f1), en regresión medimos **qué tan lejos** está la predicción del valor real.

| Métrica | Fórmula | Interpretación | Sensible a outliers |
|---------|---------|----------------|---------------------|
| **MAE** | media(\|real - pred\|) | Error promedio en la misma unidad del target | No mucho |
| **RMSE** | √media((real - pred)²) | Penaliza más los errores grandes | Sí, bastante |
| **R²** | 1 - SS_res/SS_tot | % de varianza explicada. 1.0 = perfecto, 0 = trivial | No tanto |

**¿Cuándo usar cada una?**
- Si los errores grandes son muy costosos (medicina, finanzas) → **RMSE**
- Si todos los errores pesan igual → **MAE**
- Para comparar modelos en términos relativos → **R²**

In [ ]:
# Función helper que usaremos en toda la clase
def evaluar_modelo(nombre, y_true, y_pred):
    ae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)m
    print(f'{nombre:<25} | MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}')
    return {'modelo': nombre, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

# Demo rápido con baseline
baseline_pred = np.full(len(y_te_c), y_tr_c.mean())  # siempre predice la media
evaluar_modelo('Baseline (media siempre)', y_te_c, baseline_pred)
print('\n💡 Este es el piso mínimo — cualquier modelo debe superar esto.')

---
# PARTE 4 — Pipeline + Cross Validation

### ¿Por qué Pipeline?
1. Evita data leakage en el preprocesamiento (el scaler aprende solo del train)
2. Reproducible: un solo objeto para entrenar, evaluar y predecir
3. Compatible con GridSearchCV directamente

### ¿Por qué Cross Validation?
Un solo split train/test puede darte suerte (o mala suerte). CV te da **K evaluaciones independientes** y con eso una estimación del rendimiento real del modelo con su varianza.

In [ ]:
# Dividir en train/test
# Todas las features son numéricas en este dataset

# Preprocesador: imputer + scaler para numéricas construirlo


In [ ]:
# Crea los pipelines para cada modelo



---
# PARTE 5 — Comparar los 3 modelos

Ahora sí el bloque central. Vamos a poner a competir:

| Modelo | Tipo | Strengths | Weaknesses |
|--------|------|-----------|------------|
| **LinearRegression** | Lineal | Rápido, interpretable, estable | No captura relaciones no lineales |
| **DecisionTreeRegressor** | No lineal | Captura no linealidades, interpretable | Overfitting fácil sin límite de profundidad |
| **RandomForestRegressor** | Ensemble | Robusto, reduce overfitting, muy preciso | Lento en predicción, menos interpretable |

**Hipótesis:** el precio de una casa NO es lineal respecto al ingreso — esperamos que los árboles superen a LinearRegression.

In [ ]:
# Cross Validation con R² (5 folds) para cada modelo


#Probamos el mejor modelo en el train y test set

---
# PARTE 6 — GridSearch

RandomForest ganó, pero estamos usando hiperparámetros por defecto.  
GridSearch busca sistemáticamente la mejor combinación.

**Hiperparámetros clave de RandomForest:**
- `n_estimators` — cantidad de árboles. Más árboles = más estable pero más lento
- `max_depth` — profundidad de cada árbol. Controla el overfitting
- `min_samples_split` — mínimo de muestras para hacer un split

In [ ]:
# Grid de hiperparámetros (prefijo 'model__' para referencias dentro del pipeline)
param_grid = {
    'model__n_estimators':    [100, 200],
    'model__max_depth':       [10, 20, None],
    'model__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    pipeline_rf,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print('\n✅ GridSearch terminado')
print(f'Mejores parámetros: {grid_search.best_params_}')
print(f'Mejor R² en CV:     {grid_search.best_score_:.4f}')

In [ ]:
# Ver todos los resultados del grid ordenados
cv_results = pd.DataFrame(grid_search.cv_results_)
cols = [c for c in cv_results.columns if 'param_' in c] + ['mean_test_score', 'std_test_score', 'rank_test_score']
cv_results[cols].sort_values('rank_test_score').head(8)

---
# PARTE 7 — Feature Importances

¿Qué le importa al modelo para predecir el precio?

**Recordatorio de cómo funciona en RandomForest:**  
Cada vez que un árbol hace un split usando una feature, mide cuánto redujo el MSE esa división.  
La importancia = suma acumulada de esas reducciones, promediada entre todos los árboles.  
Las importancias suman 1.0.

**Limitación:** tiene sesgo hacia features con muchos valores únicos (variables continuas).  
Por eso también veremos **Permutation Importance** como alternativa más robusta.

In [ ]:
best_model = grid_search.best_estimator_

# Extraer importancias del RandomForest dentro del pipeline
rf_model = best_model.named_steps['model']
feature_names = features_num  # en este dataset todas son numéricas

importancias = pd.Series(
    rf_model.feature_importances_,
    index=feature_names
).sort_values(ascending=True)

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
colors_imp = ['#4CBE7A' if v > importancias.mean() else '#AED6F1' for v in importancias]
bars = ax.barh(importancias.index, importancias.values, color=colors_imp, edgecolor='white')
ax.axvline(importancias.mean(), color='tomato', linestyle='--', linewidth=1.2, label='Media')
ax.set_title('Feature Importances — RandomForest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importancia (fracción de reducción de MSE)')
ax.legend()

for bar, val in zip(bars, importancias.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print('\n💡 Features por encima de la media (verde) son las más valiosas para el modelo.')
print(f'   Top feature: {importancias.idxmax()} ({importancias.max():.3f})')
print(f'   Coincide con la correlación más alta que vimos en la exploración? ✅')

In [ ]:
# Permutation Importance — más robusta, no tiene sesgo de cardinalidad
from sklearn.inspection import permutation_importance

perm_imp = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=10, random_state=42, n_jobs=-1
)

perm_series = pd.Series(
    perm_imp.importances_mean,
    index=feature_names
).sort_values(ascending=True)

# Comparación lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, titulo, color in zip(
    axes,
    [importancias, perm_series],
    ['Impurity-based Importance', 'Permutation Importance'],
    ['#4CBE7A', '#9B59B6']
):
    ax.barh(data.index, data.values, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(data.mean(), color='tomato', linestyle='--', linewidth=1.2)
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel('Importancia')

plt.suptitle('Impurity-based vs Permutation Importance', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('💡 Si el ranking coincide entre ambos métodos → más confianza en la importancia.')
print('   Si difieren mucho → posible sesgo de cardinalidad en impurity-based.')

---
# PARTE 8 — Modelo Final

Con los mejores hiperparámetros encontrados por GridSearch, entrenamos el modelo final  
sobre **todo el dataset de entrenamiento** y lo evaluamos sobre el test set que nunca tocamos.

In [ ]:
# Evaluación final sobre test set
y_pred_final = best_model.predict(X_test)

mae_final  = mean_absolute_error(y_test, y_pred_final)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_final))
r2_final   = r2_score(y_test, y_pred_final)

print('=' * 50)
print('  EVALUACIÓN FINAL — Test Set')
print('=' * 50)
print(f'  MAE:  {mae_final:.4f}  (~${mae_final*100_000:,.0f} USD de error promedio)')
print(f'  RMSE: {rmse_final:.4f}  (~${rmse_final*100_000:,.0f} USD)')
print(f'  R²:   {r2_final:.4f}  (explica el {r2_final*100:.1f}% de la varianza del precio)')
print('=' * 50)

In [ ]:
# Residual plot — detecta sesgos sistemáticos del modelo
residuales = y_test.values - y_pred_final

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pred vs Real
axes[0].scatter(y_test, y_pred_final, alpha=0.3, s=10, color='steelblue')
lims = [min(y_test.min(), y_pred_final.min()), max(y_test.max(), y_pred_final.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Predicción perfecta')
axes[0].set_xlabel('Valor real')
axes[0].set_ylabel('Valor predicho')
axes[0].set_title('Real vs Predicho', fontsize=12, fontweight='bold')
axes[0].legend()

# Residuales vs Predicho
axes[1].scatter(y_pred_final, residuales, alpha=0.3, s=10, color='steelblue')
axes[1].axhline(0, color='tomato', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Valor predicho')
axes[1].set_ylabel('Residual (real - predicho)')
axes[1].set_title('Residual Plot', fontsize=12, fontweight='bold')

plt.suptitle('Diagnóstico del modelo final', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('💡 Un buen modelo tiene residuales centrados en 0, sin patrones visibles.')
print('   Si ves una "V" o una curva → el modelo tiene sesgo sistemático.')
print('   Si ves que los residuales crecen con el valor predicho → heteroscedasticidad.')

---
# Resumen

| Paso | Qué hace | Para qué sirve |
|------|----------|----------------|
| **Data leakage** | Detectar features calculadas con el target | Evitar modelos que hacen trampa |
| **Métricas** | MAE, RMSE, R² | Medir el error en unidades reales |
| **Pipeline** | Encadenar preprocesamiento + modelo | Reproducible y seguro en producción |
| **Cross Validation** | Evaluar con K folds + barras de error | Saber si el modelo es estable |
| **Comparación** | LinearReg vs DecisionTree vs RandomForest | Elegir el enfoque correcto para el problema |
| **GridSearch** | Buscar mejores hiperparámetros | Afinar sin perder generalización |
| **Feature Importances** | Qué columnas aportan más | Explicabilidad + detección de ruido |
| **Residual Plot** | Diagnosticar el modelo final | Detectar sesgos sistemáticos |

---
### Volvamos al caso del inicio

> El modelo pasó de R² 0.95 en pruebas a predecir $80,000 para una mansión de $500,000.

La columna `precio_estimado_agencia` era una transformación ruidosa del target mismo — el modelo aprendió a usarla como atajo. En producción esa columna no existe, y el modelo colapsa.

**Regla:** antes de meter una columna al modelo, preguntá —  
*¿Esta información estaría disponible en el momento real de la predicción?*